I'm gonna use the routine imarith of IRAF for for 'cleaning' the echellograms doing:
$$\frac{image-<bias>}{(<flat>-<bias>)_N}$$
where $<flat>$ and $<bias>$ are middle values, their files are stored in their own folders.\
The $(<flat>-<bias>)$ has been normalized by dividing it for its own hightest value using the APNORMALIZE routine.

We consider first the image [XSHOO.2024-01-14T07:57:36.928.fits](XSHOO.2024-01-14T07:57:36.928.fits)

Let's now subtract the bias from the image:

In [1]:
import os
from pyraf import iraf

# 1. Load packages containing imarith and statistics
iraf.images()
iraf.imutil()

# 2.0 Define relative paths
bias_dir = "bias"
flat_dir = "flat"

# 2.1 Define file names
image_raw = "XSHOO.2024-01-14T07:57:36.928.fits"
master_bias = os.path.join(bias_dir, "Bias.fits")
raw_flat = os.path.join(flat_dir, "Flat.fits")

# Intermediate and final files
flat_sub_bias = os.path.join(flat_dir, "flat_sub_bias.fits")
flat_norm = os.path.join(flat_dir, "flat_norm.fits")
image_sub_bias = "image1_sub_bias.fits"
image_calibrated = "image1_reduced.fits"


def remove_if_exists(filepath):
    if os.path.exists(filepath):
        os.remove(filepath)


for f in [flat_sub_bias, flat_norm, image_sub_bias, image_calibrated]:
    remove_if_exists(f)

iraf.unlearn("imarith")

# 3.Subtraction of Bias from the Flat (Flat - Bias)
iraf.imarith(
    operand1=raw_flat,
    op="-",
    operand2=master_bias,
    result=flat_sub_bias,
    title="Flat debiased",
)

# 4. Subtraction of Bias from Science (Sci - Bias)
iraf.imarith(
    operand1=image_raw,
    op="-",
    operand2=master_bias,
    result=image_sub_bias,
    title="Image without Bias",
)

## 7. Cleanup intermediate files (optional)
## for f in [    ]:
##     remove_if_exists(f)


#print(f"Calibration completed: {image_calibrated}")
print(f"Done.")

Done.


I wanna now separate **echellogram peaks using APSUM Iraf package** ( [image_sub_bias.fits](image_sub_bias.fits) ---> [image_sub_bias.ms.fits](image_sub_bias.ms.fits) ).

To separate the orders of your fit image it's better to use **iraf terminal** and interactive windows.\
Run the apsum task writing an input name "file.fits" and output name like "file.ms.fits", than find the apertures by eye and number them in the correct way (select an high number of apertures at the begin, than press "." and "D" to delate them in the interactive window, "o" to rename them and "M" to add a new one).\
Now you have to fit the peaks found for every order with a polynomial, choose the order writing ":order n" (where n is a number high enough to decrease the RMS but small enough to don't include in your fit random photons that don't rapresent the image of the star). The fit can be iterated multiple times to reject the points we don't need, set the number of iteration writing ":niter n".\
You can save and exit pressing "q".\
\
For this spectra I mostly used polynomial of $4^{th}$ or $5^{th}$ order and a number of iteration between 2 and 5. The RMS was about 0.1, except the last order.\
I found 14 apertures.

I separate the flat with the same apertures of the image using apsum:

In [ ]:
from pyraf import iraf

iraf.noao()
iraf.twodspec()
iraf.apextract()

subfolder = 'flat'

iraf.apsum(
    input=f'{subfolder}/flat_sub_bias.fits',
    output=f'{subfolder}/flat1_sub_bias.ms.fits',
    apertures='',
    format='multispec',
    reference='image1_sub_bias',
    profiles='',
    interactive='no',
    find='no',
    recenter='no',
    resize='no',
    edit='no',
    trace='no',
    fittrace='no',
    extract='yes',
    extras='no',
    review='no',
    line='INDEF',
    nsum=10,
    background='none',
    weights='none',
    pfit='fit1d',
    clean='no',
    skybox=1,
    saturation='INDEF',
    readnoise=0.0,
    gain=1.0,
    lsigma=4.0,
    usigma=4.0,
    nsubaps=1,
    mode='ql'
)

twodspec/:
 apextract/     longslit/
apextract/:
 apall          apedit          apflatten       apnormalize     apscatter
 apdefault@     apfind          apmask          aprecenter      apsum
 apdemos/       apfit           apnoise         apresize        aptrace


Let's **normalize the debiased ms Flat** with **apnormalize** task (with input [flat_sub_bias.ms.fits](flat_sub_bias.ms.fits) and output [flat_norm.ms.fits](flat_norm.ms.fits)). \
It's not necessary to use it interactively.

In [3]:
#this code doesn't work so I used terminal

from pyraf import iraf

iraf.noao()
iraf.twodspec()
iraf.apextract()

subfolder = 'flat'

iraf.unlearn('apnormalize')

iraf.apnormalize(
    input=f'{subfolder}/flat1_sub_bias.ms.fits',
    output=f'{subfolder}/flat1_norm.ms.fits',
    apertures='',
    references='',
    
    # Don't active interactive mode.
    interactive='no',
    find='no',
    recenter='no',
    resize='no',
    edit='no',
    trace='no',
    fittrace='no',
    normalize='yes',
    fitspec='no', 

)

Killing IRAF task `apnormalize'


IrafError: Could not find task apnorm to get parameter apnorm.background
Failed to get parameter apnorm1.background

Let's **divide the image sub-biased by the flat normalized and sub-biased (both in the .ms.fits  format)**.
We are using XSHOOTER, it is pasted on the telescope and light is not detected trough optical fibers. 
So we could have divided the image by the normalized flat also before extracting apertures.

In [ ]:
import os
from pyraf import iraf

flat_dir = "flat"

# Intermediate and final files
flat_sub_bias_ms = os.path.join(flat_dir, "flat1_sub_bias.ms.fits")
flat_norm_ms = os.path.join(flat_dir, "flat1_norm.ms.fits")
image_sub_bias_ms = "image1_sub_bias.ms.fits"
image_calibrated_ms = "image1_calibrated.ms.fits"


#for f in [flat_sub_bias_ms, flat_norm_ms, image_sub_bias_ms, image_calibrated_ms]:
#    remove_if_exists(f)   #don't remove, files aren't created on this notebook
remove_if_exists(image_calibrated_ms)  #this one is created on this notebook.


# Division of debiased Science by normalized Flat ((Sci - Bias) / Flat_norm)
iraf.imarith(
    operand1=image_sub_bias_ms,
    op="/",
    operand2=flat_norm_ms,
    result=image_calibrated_ms,
    title="Image calibrated (Debiased Sci / Max-Norm Flat)",
)

# Cleanup intermediate files (optional)
# for f in [flat_sub_bias_ms, flat_norm_ms, image_sub_bias_ms]:
#     remove_if_exists(f)

print(f"Calibration multispectrum completed: {image_calibrated_ms}")

Calibration multispectrum completed: image1_calibrated.ms.fits
